# RF Coach Vision su Colab (GPU T4)

Il **codice** arriva da GitHub, i **file pesanti** (pesi dei modelli e video) da Google Drive.

**Preparazione, una volta sola.** Su Google Drive crea la cartella **Il mio Drive/rf_coach_vision/** e dentro:

| Sottocartella | Cosa metterci |
|---|---|
| `models/` | `tennis_yolo11.pt`, `yolo11n-pose.pt` |
| `ckpts/` | `TrackNet_best.pt`, `InpaintNet_best.pt` (da `tracknet3/ckpts` sul PC) |
| `inputs/` | i video da analizzare |
| `datasets/` | le annotazioni del *Tennis Player Actions Dataset* (servono per addestrare il classificatore dei colpi, vedi `MODELLI.md`) |

`outputs/` e `pred_result/` vengono creati da solo alla fine, con i risultati.

**Ogni volta:** menu **Runtime → Cambia tipo di runtime → GPU T4**, poi esegui le celle dall'alto in basso (la 9 è facoltativa).

## 1. Controllo GPU

In [ ]:
!nvidia-smi -L

## 2. Collega Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 3. Scarica il codice da GitHub

Il repository è **pubblico**: non serve nessun token.
Se un giorno tornasse **privato**, servirebbe un token: pannello a sinistra, icona della chiave (Secrets), un segreto chiamato `GITHUB_TOKEN` (token *fine-grained* con accesso al solo `rf_coach_vision` e permesso *Contents: Read-only*) con l'accesso al notebook attivo. La cella lo userà da sola.

In [ ]:
import os, shutil, glob, subprocess

REPO = "anrundo-2312/rf_coach_vision"
DRIVE_DIR = "/content/drive/MyDrive/rf_coach_vision"
WORK_DIR = "/content/rf_coach_vision"

# Il repo e' pubblico: il token serve solo se un giorno torna privato.
token = ""
try:
    from google.colab import userdata
    token = (userdata.get("GITHUB_TOKEN") or "").strip()
except Exception:
    pass
url = f"https://{REPO.split('/')[0]}:{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"

# Se la cella viene rieseguita, la cartella corrente e' quella che stiamo per
# cancellare (l'analisi fa %cd li'): git non partirebbe. Prima usciamo.
os.chdir("/content")
if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
r = subprocess.run(["git", "clone", "--depth", "1", url, WORK_DIR], capture_output=True, text=True,
                   env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})
if r.returncode != 0:
    raise SystemExit("Download da GitHub non riuscito:\n" + (r.stderr.replace(token, "***") if token else r.stderr))
print("codice scaricato:", sorted(os.listdir(WORK_DIR)))

## 4. Porta dal Drive modelli, video e risultati pallina già calcolati

In [ ]:
copie = [("models", ""), ("ckpts", "tracknet3/ckpts"), ("inputs", "inputs"), ("pred_result", "tracknet3/pred_result")]

for sub, dst in copie:
    src = os.path.join(DRIVE_DIR, sub)
    dst_dir = os.path.join(WORK_DIR, dst) if dst else WORK_DIR
    os.makedirs(dst_dir, exist_ok=True)
    if os.path.isdir(src):
        for f in glob.glob(os.path.join(src, "*")):
            if os.path.isfile(f):
                shutil.copy(f, dst_dir)

mancanti = [f for f in ["tennis_yolo11.pt", "yolo11n-pose.pt", "tracknet3/ckpts/TrackNet_best.pt"]
            if not os.path.exists(os.path.join(WORK_DIR, f))]
assert not mancanti, f"Mancano su Drive: {mancanti} (vedi la tabella in cima al notebook)"

print("video disponibili:", sorted(os.listdir(os.path.join(WORK_DIR, "inputs"))))

## 5. Installa le librerie (PyTorch c'è già su Colab)

In [ ]:
!pip install -q ultralytics parse

## 6. Analisi

`TRACKNET_MODE`: `"weight"` (preciso) o `"nonoverlap"` (veloce).
`FORCE`: `True` ricalcola la pallina anche se è già in `pred_result`.

In [ ]:
VIDEO = "zverev_djokovic_trim_swin_like.mp4"
TRACKNET_MODE = "weight"
FORCE = False

import re, time
p = os.path.join(WORK_DIR, "analyze.py")
s = open(p).read()
s = re.sub(r"^TRACKNET_MODE = .*$", f'TRACKNET_MODE = "{TRACKNET_MODE}"', s, flags=re.M)
s = re.sub(r"^TRACKNET_FORCE_RECOMPUTE = .*$", f"TRACKNET_FORCE_RECOMPUTE = {FORCE}", s, flags=re.M)
open(p, "w").write(s)

%cd {WORK_DIR}
t0 = time.time()
!python analyze.py inputs/{VIDEO}
print(f"\nTempo totale: {(time.time() - t0) / 60:.1f} minuti")

## 7. Classificazione dei colpi

Classifica ogni frame (dritto, rovescio, servizio, attesa), trova gli impatti dalla traiettoria della pallina, calcola le metriche di ogni colpo e crea il video con il colpo scritto in alto a sinistra: `outputs/video/<video>_combined_<modalità>_colpi.mp4`.

Il modello viene riaddestrato ogni volta dalle annotazioni su Drive (pochi secondi): così non ci sono problemi di versione di scikit-learn con il modello salvato nel repo. Se le annotazioni non ci sono, usa quello del repo.

In [ ]:
NOME = VIDEO.replace(".mp4", "")
TRACKING = f"outputs/dati/{NOME}_tracking.csv"
VIDEO_ANALISI = f"outputs/video/{NOME}_combined_{TRACKNET_MODE}.mp4"
ANNOTAZIONI = os.path.join(DRIVE_DIR, "datasets/tennis_actions/Tennis Player Actions Dataset for Human Pose Estimation/annotations")

%cd {WORK_DIR}
if os.path.isdir(ANNOTAZIONI):
    !python classificazione/addestra_colpi.py --annotazioni "{ANNOTAZIONI}"
else:
    print("Annotazioni non trovate su Drive: uso il modello salvato nel repo.")

!python classificazione/classifica_tracking.py --tracking {TRACKING}
!python classificazione/rileva_impatti.py --tracking {TRACKING}
if os.path.exists(f"outputs/dati/{NOME}_impatti.csv"):
    !python classificazione/metriche_colpi.py --tracking {TRACKING}
!python classificazione/sovrapponi_colpi.py --colpi outputs/dati/{NOME}_colpi.csv --video {VIDEO_ANALISI}

## 8. Salva i risultati su Drive
Da eseguire sempre: lo spazio di lavoro di Colab sparisce alla chiusura della sessione.

I video finiscono in *Drive/rf_coach_vision/outputs/video* (quello con i colpi termina in `_colpi.mp4`), i CSV in *outputs/dati*.
Sono salvati con il codec mp4v: si aprono con il lettore del PC, non nel browser. Per guardarli qui usa la cella 9.

In [ ]:
# I risultati sono divisi in outputs/video (i filmati) e outputs/dati (i CSV):
# la stessa struttura viene ricreata su Drive.
for sub, src in [("outputs/video", "outputs/video"),
                 ("outputs/dati", "outputs/dati"),
                 ("pred_result", "tracknet3/pred_result")]:
    dst = os.path.join(DRIVE_DIR, sub)
    os.makedirs(dst, exist_ok=True)
    for f in glob.glob(os.path.join(WORK_DIR, src, "*")):
        if os.path.isfile(f):
            shutil.copy(f, dst)
            print("salvato:", os.path.join(dst, os.path.basename(f)))

## 9. Guarda il video qui (facoltativo)

I browser non leggono il codec mp4v: questa cella fa una copia in H.264, ridotta a 1280 px, e la mostra qui sotto. La copia resta su Colab e non va su Drive. Va bene per video brevi: con video lunghi il notebook diventa pesante, meglio scaricare il file da Drive.

In [ ]:
from IPython.display import HTML
from base64 import b64encode

NOME = VIDEO.replace(".mp4", "")
src = f"{WORK_DIR}/outputs/video/{NOME}_combined_{TRACKNET_MODE}_colpi.mp4"
if not os.path.exists(src):
    src = src.replace("_colpi.mp4", ".mp4")  # classificazione non eseguita: mostro il video dell'analisi
print("mostro:", os.path.basename(src))

!ffmpeg -y -loglevel error -i "{src}" -vf scale=1280:-2 -vcodec libx264 -pix_fmt yuv420p -crf 28 /content/anteprima.mp4
dati = b64encode(open("/content/anteprima.mp4", "rb").read()).decode()
HTML(f'<video width="900" controls><source src="data:video/mp4;base64,{dati}" type="video/mp4"></video>')